In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import cartopy.crs as ccrs
from brokenaxes import brokenaxes
from adjustText import adjust_text

BASE = Path('../../data/supplementary_figures')
PKL_PATH = BASE / '../../data/supplementary_figures/slr_res/DIVA_data/Coast_with_Mang.pkl'
COASTLINE_PATH = BASE / '../../data/supplementary_figures/slr_res/global_coastal_wetland_model/Results/coastline_shp/DIVA_coastline_WGS84_Point.csv'
OUT_DIR = BASE / '../../figures/supplementary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

with PKL_PATH.open('rb') as f:
    data = pickle.load(f)

coastline = pd.read_csv(COASTLINE_PATH)
country_split = coastline['ID_Country'].str.split(' ', n=1, expand=True)
country_info = pd.DataFrame({'ClsFID': coastline['CLSFID'], 'Country': country_split[1]})

COLORS = {
    'baseline': '#cccccc',
    'growth': '#76d7c4',
    'hydro': '#f3b6b1',
    'restoration': '#ebdef7',
    'slr': '#aad6f1',
}

plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'
plt.rcParams['axes.linewidth'] = 1.0

area = data['area']
agb_present = data['agb_present']
agb_grow = data['agb_grow']
agb_245 = data['agb_245']
area_after_slr_245 = data['area_after_SLR_245'][:, 1]  # Pop5, same as the main figures
area_restoration_245 = data['area_restoration_245'][:, 0]  # 50% restoration
lat = data['lat_lon_coast'][:, 0]
lon = data['lat_lon_coast'][:, 1]

df = pd.DataFrame({
    'ClsFID': data['ClsFID'],
    'area': area,
    'agb_present': agb_present,
    'agb_grow': agb_grow,
    'agb_245': agb_245,
    'area_after_SLR_245': area_after_slr_245,
    'area_restoration_245': area_restoration_245,
})
df = df.merge(country_info, on='ClsFID', how='left')

def pgdm(values):
    return np.nansum(values) / 1e9

total_present = pgdm(agb_present * area)
total_grow = pgdm(agb_grow * area)
total_245 = pgdm(agb_245 * area)
total_245_restoration = pgdm(agb_245 * (area + area_restoration_245))
total_245_final = pgdm(agb_245 * (area_after_slr_245 + area_restoration_245))

df['present_agb_TgDM'] = df['agb_present'] * df['area'] / 1e6
df['total_change_245'] = (df['agb_245'] * (df['area_after_SLR_245'] + df['area_restoration_245']) - df['agb_present'] * df['area']) / 1e6
df['grow_change_245'] = (df['agb_grow'] * df['area'] - df['agb_present'] * df['area']) / 1e6
df['ht_change_245'] = (df['agb_245'] * df['area'] - df['agb_grow'] * df['area']) / 1e6
df['res_change_245'] = df['agb_245'] * df['area_restoration_245'] / 1e6
df['slr_change_245'] = df['agb_245'] * (df['area_after_SLR_245'] - df['area']) / 1e6

country = (
    df.groupby('Country', as_index=False)[['present_agb_TgDM', 'total_change_245', 'grow_change_245', 'ht_change_245', 'res_change_245', 'slr_change_245']]
    .sum()
    .dropna()
)

def draw_waterfall(ax):
    baseline, growth, hydro, restoration, final = total_present, total_grow, total_245, total_245_restoration, total_245_final
    width = 0.8
    ax.bar(1, baseline, width=width, color=COLORS['baseline'], edgecolor='none')
    ax.bar(2, growth, width=width, color=COLORS['growth'], edgecolor='none')
    ax.bar(2, baseline, width=width, color='white', edgecolor='white')
    ax.bar(3, hydro, width=width, color=COLORS['hydro'], edgecolor='none')
    ax.bar(3, growth, width=width, color='white', edgecolor='white')
    ax.bar(4, restoration, width=width, color=COLORS['restoration'], edgecolor='none')
    ax.bar(4, hydro, width=width, color='white', edgecolor='white')
    ax.bar(5, restoration, width=width, color=COLORS['slr'], edgecolor='none')
    ax.bar(5, final, width=width, color='white', edgecolor='none')
    ax.bar(6, final, width=width, color=COLORS['baseline'], edgecolor='none')
    deltas = [growth - baseline, hydro - growth, restoration - hydro, final - restoration]
    tops = [growth, hydro, restoration, restoration]
    bottoms = [baseline, growth, hydro, final]
    for xpos, delta, top, bottom in zip([2, 3, 4, 5], deltas, tops, bottoms):
        if delta >= 0:
            ax.text(xpos, top + 0.02, f'+{delta:.2f}', ha='center', va='bottom')
        else:
            ax.text(xpos, bottom - 0.02, f'{delta:.2f}', ha='center', va='top')
    ax.text(1, baseline * 0.82, f'{baseline:.2f}', ha='center', va='center')
    ax.text(6, final * 0.78, f'{final:.2f}', ha='center', va='center')
    ax.set_xlim(0, 7)
    ax.set_ylim(1.0, 2.5)
    ax.set_yticks(np.arange(1.0, 2.6, 0.5))
    ax.set_xticks([1, 6], ['Present-day', '2100'])
    ax.set_title('a', loc='left', fontweight='bold')
    ax.text(0.4, 2.45, 'Intermediate-warming scenario', ha='left', va='top', fontsize=16)
    ax.set_ylabel('Total AGB (Pg DM)', fontsize=16)
    ax.tick_params(axis='both')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='x', length=0)

def draw_break_marks(fig, ax_left, ax_right):
    fig.canvas.draw()
    bbox_l = ax_left.get_position()
    bbox_r = ax_right.get_position()
    y = bbox_l.y0
    dx = 0.004
    dy = 0.008
    inset = 0.001
    line_kw = dict(transform=fig.transFigure, color='black', clip_on=False, linewidth=1.2)
    fig.lines.extend([
        plt.Line2D([bbox_l.x1 - inset - dx, bbox_l.x1 - inset + dx], [y - dy, y + dy], **line_kw),
        plt.Line2D([bbox_r.x0 + inset - dx, bbox_r.x0 + inset + dx], [y - dy, y + dy], **line_kw),
    ])


def draw_s10(ax_left, ax_right):
    break_point = 190
    x_col = 'present_agb_TgDM'
    y_col = 'total_change_245'
    plot_df = country.copy()
    top_10_countries = plot_df.sort_values(by=x_col, ascending=False).head(10)['Country'].tolist()

    left = plot_df[plot_df[x_col] <= break_point]
    right = plot_df[plot_df[x_col] > break_point]
    for ax, sub in [(ax_left, left), (ax_right, right)]:
        ax.scatter(sub[x_col], sub[y_col], alpha=0.8, color='#d98c3f', edgecolors='k', s=60, zorder=10)
        ax.axhline(0, color='grey', linestyle='--', linewidth=1.2)
        ax.grid(linestyle=':', alpha=0.7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both')

    ax_left.set_xlim(0, break_point)
    ax_right.set_xlim(490, 540)
    ax_left.set_ylim(-190, 70)
    ax_right.set_ylim(-190, 70)
    ax_left.set_xticks([0, 50, 100, 150])
    ax_right.set_xticks([500])
    ax_right.tick_params(axis='y', labelleft=False, left=False)
    ax_right.spines['left'].set_visible(False)

    ax_left.set_title('b', loc='left', fontweight='bold')
    ax_left.set_ylabel('AGB Change (Tg DM)', labelpad=12)
    # ax_left.text(0.05, 0.08, 'Intermediate-warming scenario', transform=ax_left.transAxes, ha='left', va='bottom', fontsize=16)

    manual_overrides = {
        'Bangladesh': {'xytext': (-25, 35)},
        'Colombia': {'xytext': (-10, -32)},
        'Mexico': {'xytext': (0, -25)},
        'Venezuela': {'xytext': (36, -34)},
        'Brazil': {'xytext': (0, 12)},
        'Papua New Guinea': {'xytext': (68, 4)},
        'Myanmar': {'xytext': (18, 18)},
        'Malaysia': {'xytext': (22, -22)},
        'Australia': {'xytext': (18, 20)},
        'Indonesia': {'xytext': (-20, -32)},
    }

    texts_to_adjust = []
    points_to_avoid = plot_df[plot_df[x_col] < break_point]
    for _, row in plot_df.iterrows():
        if row['Country'] not in top_10_countries:
            continue
        source_ax = ax_right if row[x_col] > break_point else ax_left
        if row['Country'] in manual_overrides:
            override = manual_overrides[row['Country']]
            source_ax.annotate(
                row['Country'], xy=(row[x_col], row[y_col]),
                xytext=override.get('xytext', (5, 5)), textcoords='offset points',
                ha='center', va='bottom', fontsize=12,
                arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
            )
        elif row[x_col] < break_point:
            texts_to_adjust.append(source_ax.text(row[x_col], row[y_col], row['Country'], fontsize=12))
        else:
            source_ax.text(row[x_col], row[y_col] + 2, row['Country'], ha='center', va='bottom', fontsize=12)

    adjust_text(
        texts_to_adjust,
        x=points_to_avoid[x_col], y=points_to_avoid[y_col],
        ax=ax_left, expand_points=(2.5, 2.5),
        arrowprops=dict(arrowstyle='-', color='black', lw=0.5),
    )

def spatial_245():
    agb_present_safe = np.where(agb_present == 0, np.nan, agb_present)
    area_safe = np.where(area == 0, np.nan, area)
    growth = (agb_grow - agb_present) / agb_present_safe
    hydro = (agb_245 - agb_grow) / agb_present_safe
    restoration = (agb_245 * area_restoration_245) / (agb_present_safe * area_safe)
    slr = ((area_after_slr_245 - area) * agb_245) / (agb_present_safe * area_safe)
    total = growth + hydro + restoration + slr
    arr = np.column_stack((lat, lon, growth, hydro, restoration, slr, total))
    arr = arr[~np.isnan(arr).any(axis=1)]
    comps = {
        'lat': arr[:, 0], 'lon': arr[:, 1],
        'growth': arr[:, 2] * 100,
        'hydro': arr[:, 3] * 100,
        'restoration': arr[:, 4] * 100,
        'slr': arr[:, 5] * 100,
        'total': arr[:, 6] * 100,
    }
    stack = np.column_stack([np.abs(comps[k]) for k in ['growth', 'hydro', 'restoration', 'slr']])
    comps['leading'] = np.argmax(stack, axis=1)
    return comps

def draw_maps(ax_total, ax_lead):
    spatial = spatial_245()
    for ax in [ax_total, ax_lead]:
        ax.set_extent([-180, 180, -40, 40], crs=ccrs.PlateCarree())
        ax.coastlines(resolution='110m', linewidth=0.75, color='black', zorder=0)
        ax.set_xticks([])
        ax.set_yticks([])
    sc_total = ax_total.scatter(spatial['lon'], spatial['lat'], c=spatial['total'], cmap=plt.get_cmap('RdBu'), norm=Normalize(vmin=-100, vmax=100), s=5, transform=ccrs.PlateCarree())
    scatters = []
    specs = [
        ('growth', 0, plt.get_cmap('Greys'), Normalize(vmin=0, vmax=100), 1),
        ('hydro', 1, plt.get_cmap('bwr_r'), Normalize(vmin=-100, vmax=100), 1),
        ('restoration', 2, plt.get_cmap('Greens'), Normalize(vmin=0, vmax=100), 2),
        ('slr', 3, plt.get_cmap('PuOr'), Normalize(vmin=-100, vmax=100), 1),
    ]
    for name, idx, cmap, norm, zorder in specs:
        mask = spatial['leading'] == idx
        sc = ax_lead.scatter(spatial['lon'][mask], spatial['lat'][mask], c=spatial[name][mask], cmap=cmap, norm=norm, s=5, zorder=zorder, transform=ccrs.PlateCarree())
        scatters.append(sc)
    ax_total.set_title('c', loc='left', fontweight='bold', fontsize=18)
    ax_lead.set_title('d', loc='left', fontweight='bold', fontsize=18)
    # ax_total.text(-175, 35, 'Relative AGB change', ha='left', va='top', fontsize=13, transform=ccrs.PlateCarree())
    # ax_lead.text(-175, 35, 'Leading contributing factor', ha='left', va='top', fontsize=13, transform=ccrs.PlateCarree())
    cbar_ax = ax_total.inset_axes([0.04, 0.12, 0.20, 0.08])
    cbar = plt.colorbar(sc_total, cax=cbar_ax, orientation='horizontal')
    cbar.ax.set_title('Relative AGB change (%)', fontsize=14, pad=5)
    cbar.set_ticks([-100, 0, 100])
    cbar.ax.tick_params(labelsize=14, length=0, pad=1)
    cbar.outline.set_visible(False)
    labels = ['Potential growth\nto maturity', 'Climate and hydrology', 'Restoration', 'Sea-level rise']
    for i, (sc, lab) in enumerate(zip(scatters, labels)):
        cax = ax_lead.inset_axes([0.15, 0.4 - i * 0.12, 0.10, 0.05])
        cb = plt.colorbar(sc, cax=cax, orientation='horizontal')
        cb.set_ticks([])
        cb.outline.set_visible(False)
        cax.text(-0.065, 0.5, lab, va='center', ha='right', fontsize=11, transform=cax.transAxes)
        if i == 0:
            cb.ax.set_title('Relative change by the\nleading factor (%)', fontsize=14, pad=10, x=-0.2)

def draw_country_panels(ax_dec, ax_inc):
    dec = country.nsmallest(5, 'total_change_245')
    inc = country.nlargest(5, 'total_change_245').iloc[::-1]
    cols = ['grow_change_245', 'ht_change_245', 'res_change_245', 'slr_change_245']
    colors = [COLORS['growth'], COLORS['hydro'], COLORS['restoration'], COLORS['slr']]
    for ax, sub, text in [(ax_dec, dec, 'Largest AGB decrease countries'), (ax_inc, inc, 'Largest AGB increase countries')]:
        x = np.arange(len(sub))
        bottom_pos = np.zeros(len(sub))
        bottom_neg = np.zeros(len(sub))
        for col, color in zip(cols, colors):
            vals = sub[col].to_numpy()
            bottoms = np.where(vals >= 0, bottom_pos, bottom_neg)
            ax.bar(x, vals, bottom=bottoms, color=color, edgecolor='none', width=0.68)
            bottom_pos += np.where(vals >= 0, vals, 0)
            bottom_neg += np.where(vals < 0, vals, 0)
        ax.scatter(x, sub['total_change_245'], facecolor='none', edgecolor='black', s=70, zorder=5)
        ax.set_xticks(x, sub['Country'], rotation=35, ha='right', fontsize=16)
        ax.tick_params(axis='y', labelsize=16)
        ax.text(0.06, 0.95, text, transform=ax.transAxes, ha='left', va='top', fontsize=16)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
    ax_dec.set_title('e', loc='left', fontweight='bold', fontsize=18)
    ax_inc.set_title('f', loc='left', fontweight='bold', fontsize=18)
    ax_dec.set_ylabel('AGB change (Tg DM)', fontsize=16)
    ax_dec.set_ylim(-75, 32)
    ax_inc.set_ylim(-75, 97)


fig = plt.figure(figsize=(12, 18), constrained_layout=True)
grid = fig.add_gridspec(4, 2, height_ratios=[1, 0.5, 0.5, 1], width_ratios=[1, 1])

ax_water = fig.add_subplot(grid[0, 0])
draw_waterfall(ax_water)

s10_grid = grid[0, 1].subgridspec(1, 2, width_ratios=[0.78, 0.22], wspace=0.05)
ax_s10_left = fig.add_subplot(s10_grid[0, 0])
ax_s10_right = fig.add_subplot(s10_grid[0, 1], sharey=ax_s10_left)
draw_s10(ax_s10_left, ax_s10_right)

ax_total = fig.add_subplot(grid[1, :], projection=ccrs.PlateCarree())
ax_lead = fig.add_subplot(grid[2, :], projection=ccrs.PlateCarree())
draw_maps(ax_total, ax_lead)

ax_dec = fig.add_subplot(grid[3, 0])
ax_inc = fig.add_subplot(grid[3, 1])
draw_country_panels(ax_dec, ax_inc)

handles = [
    plt.Line2D([0], [0], marker='o', color='black', markerfacecolor='none', linestyle='none', label='Total'),
    Patch(color=COLORS['growth'], label='Potential growth to maturity'),
    Patch(color=COLORS['hydro'], label='Climate and hydrology'),
    Patch(color=COLORS['restoration'], label='Restoration'),
    Patch(color=COLORS['slr'], label='Sea-level rise')
]
ax_dec.legend(
    handles=handles, loc='lower right', frameon=False, labelspacing=0.12,
    bbox_to_anchor=(1, 0.1), handlelength=0.8, handletextpad=0.3, fontsize=16,
)

fig.canvas.draw()
draw_break_marks(fig, ax_s10_left, ax_s10_right)
bbox_l = ax_s10_left.get_position()
bbox_r = ax_s10_right.get_position()
fig.text(
    (bbox_l.x0 + bbox_r.x1) / 2, bbox_l.y0 - 0.018,
    'Present-day Mangrove AGB (Tg DM)', ha='center', va='top', fontsize=16,
)

fig.savefig(OUT_DIR / 'figS08_ssp245_intermediate_scenario.png', dpi=400, bbox_inches='tight')
# Optional PDF export removed for repository-clean reproduction.
plt.show()